## 1. Vérification du format YOLO et de `data_1.yaml`

In [ ]:
from pathlib import Path
import yaml

DATASET_DIR = Path("../trafic_data")
with open(DATASET_DIR / "data_1.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

names = cfg["names"]
nc = cfg["nc"]
print(f"Classes ({nc}) :", names)

exemple = next((DATASET_DIR / "train" / "labels").glob("*.txt"))
ligne = exemple.read_text(encoding="utf-8").splitlines()[0].split()
print("Exemple ->", exemple.name, ":", ligne)

## 2. Vérification des images, labels et nettoyage

In [ ]:
from PIL import Image

for split in ["train", "valid", "test"]:
    img_dir = DATASET_DIR / split / "images"
    lbl_dir = DATASET_DIR / split / "labels"

    imgs = {f.stem for f in img_dir.glob("*")}
    lbls = {f.stem for f in lbl_dir.glob("*.txt")}
    vides = [f for f in lbl_dir.glob("*.txt") if f.stat().st_size == 0]

    invalides = 0
    for f in lbl_dir.glob("*.txt"):
        for l in f.read_text(encoding="utf-8").splitlines():
            v = l.split()
            c, x, y, w, h = map(float, v)
            invalides += int(not (len(v) == 5 and 0 <= c < nc and 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1))

    for f in img_dir.glob("*"):
        with Image.open(f) as im:
            im.verify()

    print(f"{split.upper():5s} -> {len(imgs)} imgs | {len(lbls)} labels | sans_lbl={len(imgs-lbls)} | sans_img={len(lbls-imgs)} | vides={len(vides)} | invalides={invalides}")

## 3. Découpage train / test et doublons

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

# À exécuter une seule fois pour créer le dossier test (405 images)
test_img_dir = DATASET_DIR / "test" / "images"
test_lbl_dir = DATASET_DIR / "test" / "labels"
test_img_dir.mkdir(parents=True, exist_ok=True)
test_lbl_dir.mkdir(parents=True, exist_ok=True)

images = sorted((DATASET_DIR / "train" / "images").glob("*"))
_, test_images = train_test_split(images, test_size=405, random_state=42)

for img in test_images:
    lbl = DATASET_DIR / "train" / "labels" / f"{img.stem}.txt"
    shutil.move(str(img), str(test_img_dir / img.name))
    shutil.move(str(lbl), str(test_lbl_dir / lbl.name))

print("Split créé -> train:", len(list((DATASET_DIR / "train" / "images").glob("*"))), "| test:", len(list(test_img_dir.glob("*"))))

In [ ]:
import hashlib

hashes = {
    split: {hashlib.sha256(p.read_bytes()).hexdigest() for p in (DATASET_DIR / split / "images").glob("*")}
    for split in ["train", "valid", "test"]
}
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    print(f"Doublons exacts {a} vs {b} :", len(hashes[a] & hashes[b]))

## 4. Répartition des classes

In [ ]:
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

counts = {s: Counter() for s in ["train", "valid", "test"]}
for s in ["train", "valid", "test"]:
    for f in (DATASET_DIR / s / "labels").glob("*.txt"):
        for l in f.read_text(encoding="utf-8").splitlines():
            counts[s][int(l.split()[0])] += 1

df = pd.DataFrame({
    "classe": names,
    "train": [counts["train"][i] for i in range(nc)],
    "valid": [counts["valid"][i] for i in range(nc)],
    "test": [counts["test"][i] for i in range(nc)],
})
df["total"] = df[["train", "valid", "test"]].sum(axis=1)
df = df.sort_values("total", ascending=False)
print(df.to_string(index=False))

df.set_index("classe")[["train", "valid", "test"]].plot(kind="bar", stacked=True, figsize=(12, 4))
plt.title("Distribution des classes")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()